In [1]:
import pandas as pd

In [2]:
df = pd.read_json("databricks-dolly-15k.jsonl", lines=True)

In [3]:
print(df.shape)
display(df.head())

(15011, 4)


,instruction,context,response,category
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...,closed_qa
1,Which is a species of fish? Tope or Rope,,Tope,classification
2,Why can camels survive for long without water?,,Camels use the fat in their humps to keep them...,open_qa
3,"Alice's parents have three daughters: Amy, Jes...",,The name of the third daughter is Alice,open_qa
4,When was Tomoaki Komorida born?,Komorida was born in Kumamoto Prefecture on Ju...,"Tomoaki Komorida was born on July 10,1981.",closed_qa


In [4]:
df["instruction"] = df["instruction"].astype(str).str.strip()
before = len(df)
df = df[df["instruction"].str.len() > 0]
print("Removed empty instructions:", before - len(df))

Removed empty instructions: 0


In [5]:
non_empty_context = (df["context"].str.strip() != "").sum()
print(f"{non_empty_context} / {len(df)} rows have context")

4467 / 15011 rows have context


In [6]:
df["category"].value_counts()

,count
category,
open_qa,3742
general_qa,2191
classification,2136
closed_qa,1773
brainstorming,1766
information_extraction,1506
summarization,1188
creative_writing,709


In [7]:
print(df[df["context"].str.strip() != ""].iloc[0])
print("---")
print(df[df["context"].str.strip() == ""].iloc[0])

instruction           When did Virgin Australia start operating?
context        Virgin Australia, the trading name of Virgin A...
response       Virgin Australia commenced services on 31 Augu...
category                                               closed_qa
Name: 0, dtype: object
---
instruction    Which is a species of fish? Tope or Rope
context                                                
response                                           Tope
category                                 classification
Name: 1, dtype: object


In [8]:
def build_prompt(row):
    ctx = row["context"].strip()
    instr = row["instruction"].strip()
    if ctx:
        return f"{instr}\n\n{ctx}"
    return instr

df["prompt"] = df.apply(build_prompt, axis=1)

In [9]:
# 1. Confirm length distribution looks sane
df["prompt_wordcount"] = df["prompt"].str.split().str.len()
print(df["prompt_wordcount"].describe())

# 2. Spot check a few rows per category
for cat in df["category"].unique():
    sample = df[df["category"] == cat].iloc[0]
    print(f"--- {cat} ---")
    print(sample["prompt"])
    print()

count    15011.000000
mean        69.046166
std        154.424027
min          1.000000
25%          8.000000
50%         12.000000
75%         79.000000
max       3803.000000
Name: prompt_wordcount, dtype: float64
--- closed_qa ---
When did Virgin Australia start operating?

Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

--- classification ---
Which is a species of fish? Tope or Rope

--- open_qa ---
Why can camels survive for long without water?

--- information_extraction ---
If I have more pieces at the time of stalemate, hav

In [10]:
for pct in [90, 95, 99, 99.5, 99.9]:
    print(f"{pct}th percentile: {df['prompt_wordcount'].quantile(pct/100):.0f} words")

print("max:", df["prompt_wordcount"].max())
print("rows > 300 words:", (df["prompt_wordcount"] > 300).sum())
print("rows > 500 words:", (df["prompt_wordcount"] > 500).sum())

90th percentile: 191 words
95th percentile: 291 words
99th percentile: 651 words
99.5th percentile: 934 words
99.9th percentile: 1865 words
max: 3803
rows > 300 words: 706
rows > 500 words: 230


In [11]:
df.groupby("category")["prompt_wordcount"].describe()[["mean", "50%", "max"]]

,mean,50%,max
category,,,
brainstorming,11.616648,10.0,115.0
classification,19.022472,17.0,266.0
closed_qa,186.289904,133.0,2908.0
creative_writing,16.375176,13.0,150.0
general_qa,12.859881,8.0,841.0
information_extraction,205.622842,151.0,2360.0
open_qa,8.295831,8.0,41.0
summarization,222.656566,148.0,3803.0


In [12]:
df[(df["category"] == "general_qa") & (df["prompt_wordcount"] > 300)]

,instruction,context,response,category,prompt,prompt_wordcount
3356,What are the main reasons Parisians want elect...,,"According to the Mayor of Paris, Parisians wan...",general_qa,What are the main reasons Parisians want elect...,416
5992,Give me a one line summary of the story below:...,,An Odia sculptor successfully bargains with th...,general_qa,Give me a one line summary of the story below:...,604
10834,"The ""Garbage collection"" log of a JVM is forma...",,1. The log has 47 GC operations over a 36-minu...,general_qa,"The ""Garbage collection"" log of a JVM is forma...",841
10921,Give me a one line summary of the story below:...,,A master sculptor in Odisha is hired by the Ki...,general_qa,Give me a one line summary of the story below:...,604
11562,Give me a one line summary of the story below:...,,The Queen of Odisha invites a master stone scu...,general_qa,Give me a one line summary of the story below:...,604
12640,Summarize the following paragraph about Micros...,,Microsoft is an American software company that...,general_qa,Summarize the following paragraph about Micros...,307
13814,Summarize the following text into a sentence o...,,Following a historic first of an indictment of...,general_qa,Summarize the following text into a sentence o...,307


Remove extremely long prompts

In [13]:
MAX_WORDS = 500
before = len(df)
df = df[df["prompt_wordcount"] <= MAX_WORDS].reset_index(drop=True)
print(f"Removed {before - len(df)} rows, {len(df)} remain")

Removed 230 rows, 14781 remain


In [14]:
import re

MATH_PATTERNS = re.compile(
    r"\b(solve|equation|integral|derivative|calculate|computes?|sum of|"
    r"factor|algebra|geometry|probability|matrix|theorem|multiply|"
    r"divide|subtract|square root|exponent)\b", re.IGNORECASE
)

def is_math_like(p):
    return bool(MATH_PATTERNS.search(p))

df["is_math_like"] = df["prompt"].apply(is_math_like)
print(df["is_math_like"].sum(), "/", len(df), "flagged as math-like")

df.groupby("category")["is_math_like"].sum()

173 / 14781 flagged as math-like


,is_math_like
category,
brainstorming,3
classification,14
closed_qa,27
creative_writing,4
general_qa,17
information_extraction,52
open_qa,14
summarization,42


In [15]:
pd.set_option("display.max_colwidth", 200)
df[df["is_math_like"]].sample(15, random_state=0)[["prompt", "category"]]

,prompt,category
4612,"How can Bernoulli's principle be derived from Newton's second law of motion?\n\nBernoulli's principle is a key concept in fluid dynamics that relates pressure, speed and height. Bernoulli's princi...",summarization
12684,"You were abducted by aliens and experimented on, you were sent back to earth with the knowledge and ability to solve any problem on earth",creative_writing
4724,"What are the components of a passive crossover?\n\nA passive crossover is an electronic circuit that uses a combination of one or more resistors, inductors and capacitors. These components are com...",information_extraction
4644,Prove pythagoras theorem.,brainstorming
10854,"List the name of the areas surrounding Greece and group by directions from the passage. List the results in comma separated format.\n\nGreece, officially the Hellenic Republic, is a country in Sou...",information_extraction
8288,"What does it mean by Stochastic\n\nStochastic from Greek 'aim, guess' refers to the property of being well described by a random probability distribution. Although stochasticity and randomness are...",information_extraction
458,"Extract the criticisms that modern portfolio theory faces from this link https://en.wikipedia.org/wiki/Modern_portfolio_theory, place them in a bullet list\n\nDespite its theoretical importance, c...",information_extraction
13257,"Summarize the following paragraph about modern fighter jets\n\nCurrently the cutting edge of fighter design, fifth-generation fighters are characterized by being designed from the start to operate...",summarization
8567,"From the passage provided, determine who and when the first definition of thermodynamics was formulated.\n\nHistorically, thermodynamics developed out of a desire to increase the efficiency of ear...",information_extraction
7371,Bikes are very diverse and have different geometries based on your riding styles. For example a road bike will have a forward leaning geometry and a mountain bike will be more upright. Furthermore...,general_qa


In [16]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")  # same model used earlier for dedup

# Reference exemplars — should reflect your actual downstream task's math domain
math_exemplars = [
    "Solve for x: 2x + 5 = 13",
    "What is the derivative of f(x) = x^3 + 2x?",
    "Calculate the area of a circle with radius 7",
    "Prove that the square root of 2 is irrational",
    "What is 45 divided by 9?",
    "Find the sum of the first 10 natural numbers",
    "Simplify the expression 3(x + 4) - 2x",
    "What is the probability of rolling a 6 on a fair die?",
    "Solve the system of equations: x + y = 10, x - y = 2",
    "Compute the integral of sin(x) dx",
]

exemplar_embs = model.encode(math_exemplars, normalize_embeddings=True)
prompt_embs = model.encode(df["prompt"].tolist(), normalize_embeddings=True, show_progress_bar=True)

# max similarity to any exemplar, per prompt
sims = prompt_embs @ exemplar_embs.T   # shape (n_prompts, n_exemplars)
df["math_sim_score"] = sims.max(axis=1)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/462 [00:00<?, ?it/s]

In [17]:
df.sort_values("math_sim_score", ascending=False).head(20)[["prompt", "category", "math_sim_score"]]

,prompt,category,math_sim_score
10979,Give me a proof that the square root of 2 is irrational,creative_writing,0.923449
12942,What is a proof that there are two irrational numbers where raising one to the power of the other produces a rational number?,creative_writing,0.583917
651,Solve the following equation: y = 7x + 2 where x = 2,general_qa,0.570081
8653,Which of the numbers 1 through 10 are prime numbers?,classification,0.512229
2395,What is a derivative in finance?,open_qa,0.485296
12791,What is a derivative in finance?,open_qa,0.485296
2304,"Given a polynomial x^2 + 2x + 1, what is x?",open_qa,0.483209
2587,What is a circle?,open_qa,0.445543
13477,"Divide these numbers into prime, composite or neither. 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12.",classification,0.442894
7746,Prove that 2 + 2 = 5.,general_qa,0.440512


In [21]:
df.sort_values("math_sim_score", ascending=False).head(20)[["prompt", "category", "math_sim_score"]]

,prompt,category,math_sim_score
10979,Give me a proof that the square root of 2 is irrational,creative_writing,0.923449
12942,What is a proof that there are two irrational numbers where raising one to the power of the other produces a rational number?,creative_writing,0.583917
651,Solve the following equation: y = 7x + 2 where x = 2,general_qa,0.570081
8653,Which of the numbers 1 through 10 are prime numbers?,classification,0.512229
2395,What is a derivative in finance?,open_qa,0.485296
12791,What is a derivative in finance?,open_qa,0.485296
2304,"Given a polynomial x^2 + 2x + 1, what is x?",open_qa,0.483209
2587,What is a circle?,open_qa,0.445543
13477,"Divide these numbers into prime, composite or neither. 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12.",classification,0.442894
7746,Prove that 2 + 2 = 5.,general_qa,0.440512


In [23]:
df.sort_values("math_sim_score", ascending=False).iloc[41:60][["prompt", "category", "math_sim_score"]]

,prompt,category,math_sim_score
9832,What is the Taylor rule?,open_qa,0.360940
12788,What is ICD-9 in medical terminology?,open_qa,0.360924
7918,What is the circumference of the earth?,general_qa,0.358339
831,"Classify each of the following as either a single digit or a double digit number: 8, 4, 1, 12, 87, 65, 3, 5, 45, 97, 6, 2, 34, 71, 9, 48, 0, 7, 31, 50.",classification,0.355299
3088,What is arithmetic,general_qa,0.354680
3042,What are the 7 continents in the world,open_qa,0.351428
1599,"Which of the following are perfect square roots? Classify them as 'perfect' and 'not perfect' - 1, 23, 4, 6, 9, 10, 42, 112, 81, 100, 55, 16, 32, 25.",classification,0.349221
1097,How many triangles can be formed with 6 matchsticks of equal size without breaking or overlapping them?,brainstorming,0.346646
7979,What is rationing?,open_qa,0.341678
11559,What is the name of the first ten amendments to the Constitution?,open_qa,0.340253
